In [1]:
# Importing Libs
    
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import time

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report,accuracy_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


from sklearn.naive_bayes import GaussianNB
#from xgboost import XGBClassifier
#from lightgbm import LGBMClassifier

import warnings



sns.set_theme(style="darkgrid", rc={'figure.figsize':(10,6)})

In [2]:
FRAUD_PATH = "datasets/AIML Dataset.csv"

df = pd.read_csv(FRAUD_PATH)

categorical_features = ['type', 'nameOrig', 'nameDest', 'isFraud', 'isFlaggedFraud']
numerical_features = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

df['balanceOrigDiff'] = df['newbalanceOrig'] - df['oldbalanceOrg']
df['balanceDestDiff'] = df['newbalanceDest'] - df['oldbalanceDest']

features_model = numerical_features + ['balanceOrigDiff', 'balanceDestDiff', 'isFraud']
df_model = df[features_model]

df_model.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,balanceOrigDiff,balanceDestDiff,isFraud
0,1,9839.64,170136.0,160296.36,0.0,0.0,-9839.64,0.0,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,-1864.28,0.0,0
2,1,181.00,181.0,0.00,0.0,0.0,-181.00,0.0,1
3,1,181.00,181.0,0.00,21182.0,0.0,-181.00,-21182.0,1
4,1,11668.14,41554.0,29885.86,0.0,0.0,-11668.14,0.0,0


In [3]:
y = df_model["isFraud"]
X = df_model.drop("isFraud", axis = 1)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, stratify=y, random_state = 42)

In [5]:
scaler =  StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Random Forest Classifier

In [8]:
models = {
    'Dummy Classifier': DummyClassifier(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(),
    'Naive Bayes': GaussianNB(),
    'KNN': KNeighborsClassifier(),
}

MODEL_EVALUATION_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "f1",
    "precision",
    "recall",
    "roc_auc",
    "average_precision",
    "neg_brier_score",
    "f1_weighted",
]

In [9]:
def train_model(TRAIN_X: pd.DataFrame, 
                TRAIN_Y, 
                MODEL,
                CV):

    scores = cross_validate(
        estimator = MODEL,
        X = TRAIN_X,
        y = TRAIN_Y,
        cv = CV,
        scoring = MODEL_EVALUATION_METRICS,
    )

    return scores

In [10]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")

    results = {model_name: train_model(MODEL = classifier, TRAIN_X = X_train, 
                                       TRAIN_Y = y_train, CV = stratified_kfold) for model_name, classifier in models.items()}
               
        

In [78]:
def results_to_df(model_results):
    df = pd.DataFrame(model_results)
    df = df.T

    # list of columns
    col_list = df.columns.tolist()

    df = df.explode(col_list)

    df = df.reset_index()

    df[col_list] = df[col_list].astype(float)
    
    df.rename(columns={'index': 'models'}, inplace=True)
    return(df)

In [82]:
df_results = results_to_df(results)
df_results

,models,fit_time,score_time,test_accuracy,test_balanced_accuracy,test_f1,test_precision,test_recall,test_roc_auc,test_average_precision,test_neg_brier_score,test_f1_weighted
0,Dummy Classifier,0.328200,0.940803,0.998709,0.500000,0.000000,0.000000,0.000000,0.500000,0.001291,-0.001289,0.998064
1,Dummy Classifier,0.400971,0.917668,0.998709,0.500000,0.000000,0.000000,0.000000,0.500000,0.001291,-0.001289,0.998064
2,Dummy Classifier,0.331922,0.860094,0.998709,0.500000,0.000000,0.000000,0.000000,0.500000,0.001291,-0.001289,0.998064
3,Dummy Classifier,0.375432,0.954767,0.998709,0.500000,0.000000,0.000000,0.000000,0.500000,0.001291,-0.001289,0.998064
4,Dummy Classifier,0.436856,1.001108,0.998710,0.500000,0.000000,0.000000,0.000000,0.500000,0.001290,-0.001288,0.998066
5,Logistic Regression,3.264693,1.203579,0.999248,0.739525,0.621896,0.885852,0.479130,0.962167,0.609952,-0.000694,0.999136
6,Logistic Regression,3.643543,1.395863,0.999233,0.734307,0.612152,0.882160,0.468696,0.958897,0.612170,-0.000695,0.999116
7,Logistic Regression,2.876654,1.143718,0.999273,0.746919,0.636771,0.895899,0.493913,0.954672,0.619977,-0.000670,0.999167
8,Logistic Regression,3.162219,1.146654,0.999255,0.738660,0.623156,0.897059,0.477391,0.955283,0.616341,-0.000681,0.999141
9,Logistic Regression,2.936168,1.233847,0.999223,0.737113,0.611672,0.860979,0.474326,0.959104,0.604394,-0.000710,0.999111


In [100]:
df_results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   models                  25 non-null     object 
 1   fit_time                25 non-null     float64
 2   score_time              25 non-null     float64
 3   test_accuracy           25 non-null     float64
 4   test_balanced_accuracy  25 non-null     float64
 5   test_f1                 25 non-null     float64
 6   test_precision          25 non-null     float64
 7   test_recall             25 non-null     float64
 8   test_roc_auc            25 non-null     float64
 9   test_average_precision  25 non-null     float64
 10  test_neg_brier_score    25 non-null     float64
 11  test_f1_weighted        25 non-null     float64
dtypes: float64(11), object(1)
memory usage: 2.5+ KB


In [114]:
df_results[['test_balanced_accuracy','test_accuracy']].mean()

test_balanced_accuracy    0.740022
test_accuracy             0.997416
dtype: float64

In [118]:
df_results.groupby("models").mean().sort_values(
    "test_average_precision"
)

,fit_time,score_time,test_accuracy,test_balanced_accuracy,test_f1,test_precision,test_recall,test_roc_auc,test_average_precision,test_neg_brier_score,test_f1_weighted
models,,,,,,,,,,,
Dummy Classifier,0.374676,0.934888,0.998709,0.500000,0.000000,0.000000,0.000000,0.500000,0.001291,-0.001289,0.998064
Naive Bayes,0.941269,1.572965,0.990131,0.737086,0.113866,0.064692,0.483387,0.908744,0.080000,-0.009109,0.993900
Logistic Regression,3.176655,1.224732,0.999246,0.739305,0.621129,0.884390,0.478691,0.958025,0.612567,-0.000690,0.999134
KNN,20.673202,213.766645,0.999411,0.810178,0.730990,0.889611,0.620455,0.876654,0.672981,-0.000550,0.999358
Decision Tree,76.709565,0.973974,0.999582,0.913539,0.836446,0.845906,0.827273,0.913539,0.700104,-0.000418,0.999580
